In [1]:
import sqlite3
import pandas as pd
import numpy as np

In [2]:
# Função para realizar consultas no banco de dados SQLite
def db_query(query):
    con = sqlite3.connect('../../data/raw/database.sqlite')
    result = pd.read_sql_query(query, con)
    con.close()
    return result

In [3]:
# Função para verificar valores nulos em um DataFrame
def nulls(df):
    print("Valores nulos por coluna:")
    print(df.isnull().sum())

### Mostra as tabelas do banco de dados

In [4]:
# Listar as tabelas do banco de dados
tabelas = db_query("SELECT name FROM sqlite_master WHERE type='table';")
tabelas

,name
0,sqlite_sequence
1,Player_Attributes
2,Player
3,Match
4,League
5,Country
6,Team
7,Team_Attributes


### Mostra uma prévia das tabelas Player, Player_Attributes e Match

In [5]:
# Primeiras linhas da tabela Player
players = db_query("SELECT * FROM Player LIMIT 3;")
players

,id,player_api_id,player_name,player_fifa_api_id,birthday,height,weight
0,1,505942,Aaron Appindangoye,218353,1992-02-29 00:00:00,182.88,187
1,2,155782,Aaron Cresswell,189615,1989-12-15 00:00:00,170.18,146
2,3,162549,Aaron Doran,186170,1991-05-13 00:00:00,170.18,163


In [6]:
# Primeiras linhas da tabela Player_Attributes
players_attributes = db_query("SELECT * FROM Player_Attributes LIMIT 2;")
players_attributes

,id,player_fifa_api_id,player_api_id,date,overall_rating,potential,preferred_foot,attacking_work_rate,defensive_work_rate,crossing,...,vision,penalties,marking,standing_tackle,sliding_tackle,gk_diving,gk_handling,gk_kicking,gk_positioning,gk_reflexes
0,1,218353,505942,2016-02-18 00:00:00,67,71,right,medium,medium,49,...,54,48,65,69,69,6,11,10,8,8
1,2,218353,505942,2015-11-19 00:00:00,67,71,right,medium,medium,49,...,54,48,65,69,69,6,11,10,8,8


In [7]:
# Primeiras linhas da tabela Match
matches = db_query("SELECT * FROM Match LIMIT 2;")
matches

,id,country_id,league_id,season,stage,date,match_api_id,home_team_api_id,away_team_api_id,home_team_goal,...,SJA,VCH,VCD,VCA,GBH,GBD,GBA,BSH,BSD,BSA
0,1,1,1,2008/2009,1,2008-08-17 00:00:00,492473,9987,9993,1,...,4.0,1.65,3.40,4.50,1.78,3.25,4.00,1.73,3.40,4.2
1,2,1,1,2008/2009,1,2008-08-16 00:00:00,492474,10000,9994,0,...,3.8,2.00,3.25,3.25,1.85,3.25,3.75,1.91,3.25,3.6


In [8]:
# Query para obter a evolução dos jogadores ao longo do tempo, ordenada por player_api_id e data
evolution = db_query("SELECT " \
"Player.player_api_id, Player.player_name, Player.birthday, " \
"Player_Attributes.overall_rating, Player_Attributes.potential, Player_Attributes.date " \
"FROM Player " \
"INNER JOIN Player_Attributes " \
"ON Player.player_api_id = Player_Attributes.player_api_id " \
"ORDER BY Player.player_api_id, Player_Attributes.date;")
evolution

,player_api_id,player_name,birthday,overall_rating,potential,date
0,2625,"Patryk Rachwal,18",1981-01-27 00:00:00,63.0,64.0,2007-02-22 00:00:00
1,2625,"Patryk Rachwal,18",1981-01-27 00:00:00,63.0,64.0,2007-08-30 00:00:00
2,2625,"Patryk Rachwal,18",1981-01-27 00:00:00,60.0,64.0,2008-08-30 00:00:00
3,2625,"Patryk Rachwal,18",1981-01-27 00:00:00,60.0,64.0,2010-08-30 00:00:00
4,2625,"Patryk Rachwal,18",1981-01-27 00:00:00,59.0,63.0,2011-02-22 00:00:00
...,...,...,...,...,...,...
183973,750435,Rees Greenwood,1996-01-20 00:00:00,56.0,70.0,2016-02-04 00:00:00
183974,750435,Rees Greenwood,1996-01-20 00:00:00,56.0,70.0,2016-02-11 00:00:00
183975,750435,Rees Greenwood,1996-01-20 00:00:00,60.0,74.0,2016-04-14 00:00:00
183976,750584,Alexandre Azevedo,1997-01-28 00:00:00,58.0,66.0,2007-02-22 00:00:00


In [9]:
nulls(evolution)

Valores nulos por coluna:
player_api_id       0
player_name         0
birthday            0
overall_rating    836
potential         836
date                0
dtype: int64


In [10]:
# Expressão regular para validar os nomes dos jogadores (utilizada para identificar nomes que não seguem o padrão esperado)
pattern = r"^[a-zA-Zà-üÀ-Ü]+['\-\.]?[a-zA-Zà-üÀ-Ü]*(?:\s+[a-zA-Zà-üÀ-Ü]+['\-\.]?[a-zA-Zà-üÀ-Ü]*)*$"

### O código abaixo foi utilizado para identificar padões incorretos na coluna player_name que contem os nomes dos jogadores.

In [11]:
# Identificar nomes que não correspondem ao padrão (utilizada para identificar nomes que não seguem o padrão esperado)
invalid_names = evolution[~evolution['player_name'].str.contains(pattern, na=False, regex=True)]
#print(invalid_names['player_name'])
invalid_names['player_name'].to_csv('nomes_invalidos.txt', index=False, header=False)